In [ ]:
import sys
from pathlib import Path
import numpy as np
import cv2
from ultralytics import YOLO

# 1) 固定 repo_root（notebooks/ 的上一层）
repo_root = Path.cwd().parent
print("repo_root =", repo_root)

# 2) 让 Python 能 import src/zola_robot
sys.path.insert(0, str(repo_root / "src"))

from zola_robot.features.feature_extractor import FeatureExtractor

# 3) 自动找到最新 best.pt（不再手写 train2）
def find_latest_best_pt(root: Path) -> Path:
    candidates = list((root / "runs" / "pose").glob("train*/weights/best.pt"))
    if not candidates:
        raise FileNotFoundError("No best.pt found under runs/pose/train*/weights/")
    # 以文件修改时间排序，取最新的
    return max(candidates, key=lambda p: p.stat().st_mtime)

weights_path = find_latest_best_pt(repo_root)
print("Using weights:", weights_path)

# 4) SOURCE 用 repo_root 拼绝对路径（不会因为 cwd 改变而坏）
SOURCE = repo_root / "notebooks" / "sample_photos" / "zolame.jpg"   # or a video path
# SOURCE = "camera"
MAX_FRAMES = 200
SHOW_EVERY = 5

model = YOLO(str(weights_path))
fe = FeatureExtractor()

print("Feature dim:", len(fe.get_feature_names()))
print("Feature names:", fe.get_feature_names())


In [ ]:
def open_source(source):
    if isinstance(source, Path):
        source = str(source)

    p = Path(source)
    if source in ("camera", "0"):
        return "camera", cv2.VideoCapture(0)
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]:
        return "image", str(p)
    return "video", cv2.VideoCapture(str(p))


def infer_one_frame(frame_bgr):
    result = model(frame_bgr, verbose=False)[0]
    overlay = result.plot()

    if result.keypoints is None or len(result.keypoints.xy) == 0:
        return overlay, None, None

    xy = result.keypoints.xy[0].cpu().numpy()     # (24,2)
    conf = result.keypoints.conf[0].cpu().numpy() # (24,) in [0,1]

    # 推荐：直接把 conf 当作 v（0~1），不要映射成 0/1/2（更标准也更连续）
    keypoints_24x3 = np.concatenate([xy, conf[:, None]], axis=1)  # (24,3) [x,y,v]

    feats_dict = fe.extract_frame_features(keypoints_24x3.tolist())
    feats_vec  = fe.extract_frame_vector(keypoints_24x3.tolist())  # fixed order vector

    return overlay, feats_dict, feats_vec


def show_bgr(img_bgr, title=""):
    import matplotlib.pyplot as plt
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 6))
    plt.title(title)
    plt.imshow(img_rgb)
    plt.axis("off")
    plt.show()


mode, cap_or_path = open_source(SOURCE)

if mode == "image":
    img = cv2.imread(cap_or_path)
    overlay, feats_dict, feats_vec = infer_one_frame(img)

    show_bgr(overlay, "YOLO pose overlay (image)")
    if feats_dict is None:
        print("No dog detected.")
    else:
        print("Fixed-order features:")
        for name, val in zip(fe.get_feature_names(), feats_vec):
            print(f"{name:20s} {val}")
else:
    cap = cap_or_path
    if not cap.isOpened():
        raise RuntimeError("Cannot open video/camera source")

    frame_idx = 0
    while frame_idx < MAX_FRAMES:
        ret, frame = cap.read()
        if not ret:
            break

        overlay, feats_dict, feats_vec = infer_one_frame(frame)

        if frame_idx % SHOW_EVERY == 0:
            show_bgr(overlay, f"frame {frame_idx}")
            if feats_dict is None:
                print("No dog detected.")
            else:
                print("vector shape:", feats_vec.shape)
                print("visible_ratio:", feats_dict.get("visible_ratio"))
                print("torso_len:", feats_dict.get("torso_len"))
                print("head_yaw:", feats_dict.get("head_yaw"))
                print("-"*60)

        frame_idx += 1

    cap.release()
    print("Done.")


/Users/shiruiz/Github/runs/pose/train2/weights/best.pt False
